<a href="https://colab.research.google.com/github/Superfawll/ml_performance_playground/blob/main/sharded_matmul_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax
from jax import numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P

L = 6 # num_layers

Auto = jax.sharding.AxisType.Auto
mesh = jax.make_mesh(axis_shapes=(2, 2), axis_names=('X', 'Y'), axis_types=(Auto, Auto))

/home/superfall/.local/lib/python3.10/site-packages/jax/_src/cloud_tpu_init.py:84: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
f = lambda A, B: jnp.einsum('ij,jk->ik', A, B)
g = lambda Zi: jax.nn.softmax(Zi)

@jax.jit
def forward_pass(A, weights):
  def step(A, pair):
    win, wout = pair
    return g(f(f(A, win), wout)), None

  Z_final, _ = jax.lax.scan(step, A, weights)
  return Z_final

In [ ]:
key = jax.random.key(0)
A_small = jax.random.normal(key, shape=(66_036, 8_192)) # BD

wins = []
wouts = []

for layer in range(L):
    key, k1, k2 = jax.random.split(key, 3)
    wins.append(jax.random.normal(k1, (8_192, 16_384)))
    wouts.append(jax.random.normal(k2, (16_384, 8_192)))

wins = jnp.stack(wins)    # (L, 8_192, 16_384)
wouts = jnp.stack(wouts)  # (L, 16_384, 8_192)

In [ ]:
a = jax.device_put(A_small, NamedSharding(mesh, P(None, None)))

print("no_sharding")

logdir = "/tmp/xprof-small"
with jax.profiler.trace(logdir):
  with jax.profiler.TraceAnnotation("no_sharding"):
    forward_pass(a, (wins, wouts)).block_until_ready()

no_sharding


2026-10-01 14:29:17.772426: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


HloModule jit_forward_pass, is_scheduled=true, entry_computation_layout={(f32[33018,4096]{1,0:T(8,128)}, f32[6,4096,8192]{2,1,0:T(8,128)}, f32[6,8192,4096]{2,1,0:T(8,128)})->f32[33018,4096]{1,0:T(8,128)}}, allow_spmd_sharding_propagation_to_parameters={false,false,false}, allow_spmd_sharding_propagation_to_output={true}, num_partitions=4

%fused_computation.8.clone.clone (param_0.25: bf16[6,4096,8192], param_1.25: s32[]) -> bf16[1,4096,8192] {
  %param_0.25 = bf16[6,4096,8192]{2,1,0:T(8,128)(2,1)} parameter(0)
  %param_1.25 = s32[]{:T(128)} parameter(1)
  %constant.63 = s32[]{:T(128)} constant(0), metadata={op_name="jit(forward_pass)/jit(main)/jit(forward_pass)/while/body/dynamic_slice" source_file="/tmp/ipykernel_35088/1901076994.py" source_line=12}
  ROOT %dynamic-slice.23 = bf16[1,4096,8192]{2,1,0:T(8,128)(2,1)S(1)} dynamic-slice(%param_0.25, %param_1.25, %constant.63, %constant.63), dynamic_slice_sizes={1,4096,8192}, metadata={op_name="jit(forward_pass)/jit(main)/jit(forward_pass)/

2026-10-01 14:30:16.390525: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [ ]:
a = jax.device_put(A_small, NamedSharding(mesh, P('X', None)))
logdir = "/tmp/xprof-small"
with jax.profiler.trace(logdir):
  with jax.profiler.TraceAnnotation("x_shard_act_batches"):
    forward_pass(a, (wins, wouts)).block_until_ready()

In [ ]:
print("Ix, Jy sharding for A, Jy for W")

logdir = "/tmp/xprof-small"
with jax.profiler.trace(logdir):
  a = jax.device_put(A_small, NamedSharding(mesh, P(None, ('X', 'Y'))))
  with jax.profiler.TraceAnnotation("xy_shard_act_batches"):
    forward_pass(a, (wins, wouts)).block_until_ready()

Ix, Jy sharding for A, Jy for W


2026-10-01 14:21:43.148690: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-10-01 14:21:45.850446: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [ ]:

print("Ix sharding for A, Jy for W")

logdir = "/tmp/xprof-small"
with jax.profiler.trace(logdir):
  a = jax.device_put(A_small, NamedSharding(mesh, P('X', None)))
  wins  = jax.device_put(wins,  NamedSharding(mesh, P(None, 'X', None)))
  wouts = jax.device_put(wouts, NamedSharding(mesh, P(None, None, 'X')))
  with jax.profiler.TraceAnnotation("fsdp"):
    forward_pass(a, (wins, wouts)).block_until_ready()

Ix sharding for A, Jy for W


2026-10-01 14:21:11.252358: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-10-01 14:21:20.075309: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [ ]:

print("Ix sharding for A, Jy for W")

logdir = "/tmp/xprof-small"
with jax.profiler.trace(logdir):
  a = jax.device_put(A_small, NamedSharding(mesh, P(None, None)))
  wins  = jax.device_put(wins,  NamedSharding(mesh, P(None, None, 'Y')))
  wouts = jax.device_put(wouts, NamedSharding(mesh, P(None, 'Y', None)))
  with jax.profiler.TraceAnnotation("tp"):
    forward_pass(a, (wins, wouts)).block_until_ready()

# lowered = jax.jit(forward_pass).lower(a, (wins, wouts))
# compiled_hlo = lowered.compile().as_text()
# print(compiled_hlo)

Ix sharding for A, Jy for W


2026-10-01 14:20:56.570564: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-10-01 14:20:59.579437: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [ ]:
print("Ix sharding for A, Jy for W")

logdir = "/tmp/xprof-small"
with jax.profiler.trace(logdir):
  a = jax.device_put(A_small, NamedSharding(mesh, P('X', 'Y')))
  wins  = jax.device_put(wins,  NamedSharding(mesh, P(None, 'X', 'Y')))
  wouts = jax.device_put(wouts, NamedSharding(mesh, P(None, 'Y', 'X')))
  with jax.profiler.TraceAnnotation("fsdp_and_tp"):
    forward_pass(a, (wins, wouts)).block_until_ready()

lowered = jax.jit(forward_pass).lower(a, (wins, wouts))
compiled_hlo = lowered.compile().as_text()
print(compiled_hlo)

Ix sharding for A, Jy for W


2026-10-01 14:21:35.014607: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-10-01 14:21:37.653023: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


HloModule jit_forward_pass, is_scheduled=true, entry_computation_layout={(f32[33018,4096]{1,0:T(8,128)}, f32[6,4096,8192]{2,1,0:T(8,128)}, f32[6,8192,4096]{2,1,0:T(8,128)})->f32[33018,4096]{1,0:T(8,128)}}, allow_spmd_sharding_propagation_to_parameters={false,false,false}, allow_spmd_sharding_propagation_to_output={true}, num_partitions=4

%fused_computation.8.clone.clone (param_0.25: bf16[6,4096,8192], param_1.25: s32[]) -> bf16[1,4096,8192] {
  %param_0.25 = bf16[6,4096,8192]{2,1,0:T(8,128)(2,1)} parameter(0)
  %param_1.25 = s32[]{:T(128)} parameter(1)
  %constant.63 = s32[]{:T(128)} constant(0), metadata={op_name="jit(forward_pass)/jit(main)/jit(forward_pass)/while/body/dynamic_slice" source_file="/tmp/ipykernel_35088/1901076994.py" source_line=12}
  ROOT %dynamic-slice.23 = bf16[1,4096,8192]{2,1,0:T(8,128)(2,1)S(1)} dynamic-slice(%param_0.25, %param_1.25, %constant.63, %constant.63), dynamic_slice_sizes={1,4096,8192}, metadata={op_name="jit(forward_pass)/jit(main)/jit(forward_pass)/